In [ ]:
from pathlib import Path
import numpy as np
from ultralytics import YOLO
import torch
# DEVICE = "cpu"
ROOT = Path(".").resolve()
DATASET_DIR = ROOT / "datasets/field"

print("Project root:", ROOT)
print("Device:", DEVICE)

/home/mohammad-amin/footballenv/lib/python3.13/site-packages/ultralytics/yolo/utils/checks.py:16: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  import pkg_resources as pkg


Project root: /home/mohammad-amin/Desktop/footbal anaylsis
Device: cpu


In [2]:
print(np.__version__)
print(hasattr(np, "trapz"))


1.26.4
True


In [3]:
DATASET_DIR

PosixPath('/home/mohammad-amin/Desktop/footbal anaylsis/datasets/field')

In [4]:
yaml_path = DATASET_DIR/"data.yaml"

In [5]:
original_load = torch.load
def patched_load(*args, **kwargs):
    kwargs['weights_only'] = False
    return original_load(*args, **kwargs)
torch.load = patched_load

In [6]:
model  = YOLO("yolo11n-pose.pt")

AttributeError: Can't get attribute 'C3k2' on <module 'ultralytics.nn.modules.block' from '/home/mohammad-amin/footballenv/lib/python3.13/site-packages/ultralytics/nn/modules/block.py'>

In [7]:
metrics = model.train(
    data=str(yaml_path),
    epochs=50,
    imgsz=640,
    batch=4,
    device=DEVICE,
    workers=2,  # set to 2–4 on Linux/macOS if stable
    project=str(ROOT / "runs_field"),
    name="detect_custom",
    exist_ok=True,
)


Ultralytics YOLOv8.0.100 🚀 Python-3.13.9 torch-2.11.0+cu130 CPU
yolo/engine/trainer: task=detect, mode=train, model=yolov8n.pt, data=/home/mohammad-amin/Desktop/footbal anaylsis/datasets/field/data.yaml, epochs=50, patience=50, batch=4, imgsz=640, save=True, save_period=-1, cache=False, device=cpu, workers=2, project=/home/mohammad-amin/Desktop/footbal anaylsis/runs_field, name=detect_custom, exist_ok=True, pretrained=False, optimizer=SGD, verbose=True, seed=0, deterministic=True, single_cls=False, rect=False, cos_lr=False, close_mosaic=0, resume=False, amp=True, overlap_mask=True, mask_ratio=4, dropout=0.0, val=True, split=val, save_json=False, save_hybrid=False, conf=None, iou=0.7, max_det=300, half=False, dnn=False, plots=True, source=None, show=False, save_txt=False, save_conf=False, save_crop=False, show_labels=True, show_conf=True, vid_stride=1, line_width=None, visualize=False, augment=False, agnostic_nms=False, classes=None, retina_masks=False, boxes=True, format=torchscript, k

ValueError: not enough values to unpack (expected 3, got 0)

In [ ]:
run_dir = ROOT / "runs" / "detect_custom"
best_weights = run_dir / "weights" / "best.pt"

val_model = YOLO(str(best_weights))
metrics = val_model.val(data=str(yaml_path), device=DEVICE, workers=2)
print(metrics)

Ultralytics YOLOv8.0.100 🚀 Python-3.13.9 torch-2.11.0+cu130 CPU
Model summary (fused): 168 layers, 3006428 parameters, 0 gradients, 8.1 GFLOPs
val: Scanning /home/mohammad-amin/Desktop/footbal anaylsis/datasets/players/labels/val.cache... 38 images, 0 backgrounds, 0 corrupt: 100%|██████████| 38/38 [00:00<?, ?it/s]
/home/mohammad-amin/footballenv/lib/python3.13/site-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100%|██████████| 3/3 [00:01<00:00,  1.79it/s]
                   all         38        905     0.0133      0.135     0.0116    0.00512
                  ball         38         35          0          0          0          0
            goalkeeper         38         27    0.00361      0.296    0.00442    0.00109
                player         38    

ultralytics.yolo.utils.metrics.DetMetrics object with attributes:

ap_class_index: array([0, 1, 2, 3])
box: ultralytics.yolo.utils.metrics.Metric object
confusion_matrix: <ultralytics.yolo.utils.metrics.ConfusionMatrix object at 0x70872bc34750>
fitness: 0.00576513287590013
keys: ['metrics/precision(B)', 'metrics/recall(B)', 'metrics/mAP50(B)', 'metrics/mAP50-95(B)']
maps: array([          0,   0.0010914,    0.019106,  0.00028501])
names: {0: 'ball', 1: 'goalkeeper', 2: 'player', 3: 'referee'}
plot: True
results_dict: {'metrics/precision(B)': 0.013259280140684367, 'metrics/recall(B)': 0.13521987325745558, 'metrics/mAP50(B)': 0.011565647257164177, 'metrics/mAP50-95(B)': 0.005120631277981902, 'fitness': 0.00576513287590013}
save_dir: PosixPath('runs/detect/val2')
speed: {'preprocess': 0.8943331869024979, 'inference': 28.499151531018708, 'loss': 6.274173134251644e-05, 'postprocess': 6.314258826406379}


In [ ]:
PREDICT_SOURCE = str(DATASET_DIR / "images" / "val")  # folder of .jpg images

pred_model = YOLO(str(best_weights))
pred_model.predict(
    source=PREDICT_SOURCE,
    device=DEVICE,
    conf=0.25,
    save=True,
    project=str(ROOT / "runs"),
    name="predict_custom",
    exist_ok=True,
)


image 1/38 /home/mohammad-amin/Desktop/footbal anaylsis/datasets/players/images/val/08fd33_3_1_png.rf.6f25c835bf6d1828dcf584e5969b1f58.jpg: 384x640 (no detections), 34.3ms
image 2/38 /home/mohammad-amin/Desktop/footbal anaylsis/datasets/players/images/val/08fd33_3_3_png.rf.128b8280598b9931fdeeed42b5be4c51.jpg: 384x640 (no detections), 28.7ms
image 3/38 /home/mohammad-amin/Desktop/footbal anaylsis/datasets/players/images/val/08fd33_9_8_png.rf.cc61e7ba09940f4606e4464dd621fe2f.jpg: 384x640 (no detections), 31.6ms
image 4/38 /home/mohammad-amin/Desktop/footbal anaylsis/datasets/players/images/val/121364_7_9_png.rf.bd5ceb93233525ef03fac0eae292f5ed.jpg: 384x640 (no detections), 29.2ms
image 5/38 /home/mohammad-amin/Desktop/footbal anaylsis/datasets/players/images/val/121364_9_2_png.rf.400d952c966048709aa5b421889a4dba.jpg: 384x640 (no detections), 30.6ms
image 6/38 /home/mohammad-amin/Desktop/footbal anaylsis/datasets/players/images/val/121364_9_3_png.rf.175a041e339a6a4c918ae7f7141461aa.jpg:

[ultralytics.yolo.engine.results.Results object with attributes:
 
 boxes: ultralytics.yolo.engine.results.Boxes object
 keypoints: None
 keys: ['boxes']
 masks: None
 names: {0: 'ball', 1: 'goalkeeper', 2: 'player', 3: 'referee'}
 orig_img: array([[[201, 253, 253],
         [204, 254, 252],
         [208, 255, 252],
         ...,
         [108, 102,  91],
         [101,  95,  84],
         [ 99,  93,  82]],
 
        [[199, 250, 252],
         [204, 254, 254],
         [208, 255, 253],
         ...,
         [ 99,  93,  82],
         [ 93,  87,  76],
         [ 92,  86,  75]],
 
        [[198, 246, 252],
         [206, 253, 255],
         [212, 255, 255],
         ...,
         [ 96,  90,  79],
         [ 93,  87,  76],
         [ 94,  88,  77]],
 
        ...,
 
        [[ 45,  39,  34],
         [ 42,  38,  33],
         [ 43,  37,  32],
         ...,
         [102, 103,  94],
         [106, 107,  97],
         [ 97,  98,  88]],
 
        [[ 25,  21,  16],
         [ 24,  21,  16],
